In [1]:
import pyreadstat
import pandas as pd
from pathlib import Path
import os
import csv

# Data Processing

The following file converts the original data file into a format compatible with Python libraries, with the objective of eassing the analysis

In [2]:
df, meta = pyreadstat.read_sav("../object location data/Definitieve data file Data masterthesis GOED.sav")


In [3]:
df.head()

,ID,sekse,groep,leeftijd,opleiding,RBMT,Stroop_k1,Stroop_k2,Stroop_k3,IF_Stroop,...,filter_$,Diff_score_NT,Diff_score_sRI,Diff_score_dRI,Diff_score_delay,perc_diffscore_NT,perc_diffscore_sRI,perc_diffscore_dRI,perc_diffscore_delay,Nulscores
0,1.0,1.0,0.0,58.0,6.0,55.0,54.0,69.0,96.0,109.0,...,1.0,-0.033016,0.064343,0.054757,-0.118331,-3.301575,6.434347,5.475653,-11.833126,0.0
1,2.0,0.0,0.0,65.0,3.0,NaN,104.0,132.0,350.0,NaN,...,1.0,0.881896,0.515170,0.189972,-0.060228,88.189608,51.516973,18.997194,-6.022760,0.0
2,3.0,0.0,0.0,54.0,5.0,58.0,46.0,100.0,134.0,147.0,...,1.0,0.301841,0.318815,1.085214,0.689474,30.184108,31.881517,108.521447,68.947383,0.0
3,4.0,0.0,0.0,58.0,5.0,54.0,60.0,79.0,110.0,125.0,...,1.0,0.160216,0.197832,0.250294,-0.099256,16.021604,19.783162,25.029393,-9.925591,0.0
4,5.0,0.0,0.0,53.0,3.0,53.0,68.0,80.0,154.0,132.0,...,1.0,-0.006237,0.161231,-0.087561,0.472483,-0.623707,16.123073,-8.756147,47.248282,0.0


In [4]:
df_meta = pd.DataFrame({
    "name": meta.column_names,
    "label": meta.column_labels,
    "measure": [meta.variable_measure.get(col) for col in meta.column_names],
})


df_meta.head()

,name,label,measure
0,ID,Participantnr.,nominal
1,sekse,Sekse,nominal
2,groep,Participantgroep,nominal
3,leeftijd,Leeftijd,scale
4,opleiding,Opleidingsniveau,ordinal


In [12]:
print('Permutations: ', pd.unique(df['permut.6']))
print('Substitutions: ', pd.unique(df['subst.6']))
print('Swaps: ', pd.unique(df['swap.6']))

Permutations:  ['5471392806' '7128450639' '4893256017' '5403276198' '3758149620'
 '5964210783' '5712460839' '7024593861' '2438791506' '9723458610'
 '0243156789' '5126437809' '2607415389' '8193456702' '8130459762'
 '0193456782' '0192458763' '6834951720' '0423157689' '0483156729'
 '0647825319' '9046581327' '4275308916' '5189602743' '0123459768'
 '5198426730' '5193427680' '5628190734' '7692450813' '9180526743'
 '0123456789' '5234810679' '0124358769' '7123456890' '3195827640'
 '0132458769' '5312897604' '9421758360' nan]
Substitutions:  [10.  3.  5.  6.  8.  9.  4.  0.  7. nan]
Swaps:  [ 0.  1.  2. nan]


## Extract patient information

In [8]:
for i in df.columns:
    print(i)

ID
sekse
groep
leeftijd
opleiding
RBMT
Stroop_k1
Stroop_k2
Stroop_k3
IF_Stroop
Tscore_IF_Stroop
percent_IF_Stroop
NLV
TIQ_WAIS
VBI_WAIS
PRI_WAIS
WG_WAIS
PF_LLT
percent_PF_LLT
leerind_LLT
percent_leerind_LLT
no_obj.1
no_obj.2
no_obj.3
no_obj.4
no_obj.5
no_obj.6
no_obj.7
no_obj.8
no_obj.9
no_obj.10
no_obj.11
no_obj.12
no_obj.13
no_obj.14
no_obj.15
no_obj.16
no_obj.17
no_obj.18
no_obj.19
no_obj.20
time.1
time.2
time.3
time.4
time.5
time.6
time.7
time.8
time.9
time.10
time.11
time.12
time.13
time.14
time.15
time.16
time.17
time.18
time.19
time.20
abs_err.1
abs_err.2
abs_err.3
abs_err.4
abs_err.5
abs_err.6
abs_err.7
abs_err.8
abs_err.9
abs_err.10
abs_err.11
abs_err.12
abs_err.13
abs_err.14
abs_err.15
abs_err.16
abs_err.17
abs_err.18
abs_err.19
abs_err.20
bestft.1
bestft.2
bestft.3
bestft.4
bestft.5
bestft.6
bestft.7
bestft.8
bestft.9
bestft.10
bestft.11
bestft.12
bestft.13
bestft.14
bestft.15
bestft.16
bestft.17
bestft.18
bestft.19
bestft.20
permut.1
permut.2
permut.3
permut.4
permut.5
perm

In [5]:
# Generate a folder per participant
base_dir = Path("../data")

for pid in pd.unique(df["ID"]):
    participant_dir = base_dir / f"participant {int(pid)}"
    participant_dir.mkdir(parents=True, exist_ok=True)

## Extract trial information

In [56]:
# Extract task information per trial
input_path = "../object location data/Alle data GOED/"
output_path = "../data/"

files = sorted(os.listdir(input_path))

for f in files: 
    if '.ipynb_checkpoints' in f: 
        continue
        
    name_file = f.split('.')[0]
    participant_ID = int(name_file[-3:-1])
    
    with open(input_path + f, 'r') as file:
        lines = [line.strip() for line in file if line.strip()]

    # Process each trial 
    trial_id = 0
    trial_rows = []
    encoding_time = 0
    
    i = 0
    while i < len(lines):
        line = lines[i]
        print(i, trial_rows)
        # Detect start of a new trial 
        if line == "0,2,0,0" and i > 1: 
            # Save data 
            out_file = os.path.join(output_path + f'participant {participant_ID}', f"{name_file}_trial_{trial_id:02d}.csv")
            with open(out_file, "w", newline="") as f_out:
                writer = csv.DictWriter(f_out, fieldnames=trial_rows[0].keys())
                writer.writeheader()
                writer.writerows(trial_rows)
            trial_rows = []

            if i > 0: # Save encoding time of the new trial 
                prev_fields = lines[i-1].split(",")
                encoding_time = int(float(prev_fields[-1]))
                
            trial_id += 1
            i += 1
            continue
            
        fields = [x.strip().strip('"') for x in line.split(",")]
        # Detect object defintion line 
        if len(fields) == 4 and ".ico" in str(fields[-1].lower()):
            print('hola')
            object_id = int(fields[0])
            target_x = float(fields[1])
            target_y = float(fields[2])
            object_name = fields[3].strip()
            placements = []
            i += 1

            # Read placements until -99 
            while i < len(lines):
                if lines[i] == "-99,-99,-99":
                    break
                px, py, t = lines[i].split(",")
                placements.append((float(px), float(py), int(float(t))))
                i += 1

            # Save trial-object data
            if placements:
                final_x, final_y, time_ms = placements[-1]
                trial_rows.append({
                    "participant": participant_ID,
                    "trial": trial_id,
                    "object_id": object_id,
                    "object_name": object_name,
                    "target_x": target_x,
                    "target_y": target_y,
                    "final_x": final_x,
                    "final_y": final_y,
                    "time_ms": time_ms,
                    "encoding_time": encoding_time
                })

        i += 1

    # Save last trial before finishing rows 
    if trial_rows:
        out_file = os.path.join(output_path, f"{name_file}_trial_{trial_id:02d}.csv")
        with open(out_file, "w", newline="") as f_out:
            writer = csv.DictWriter(f_out, fieldnames=trial_rows[0].keys())
            writer.writeheader()
            writer.writerows(trial_rows)


IOPub data rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_data_rate_limit`.

Current values:
ServerApp.iopub_data_rate_limit=1000000.0 (bytes/sec)
ServerApp.rate_limit_window=3.0 (secs)

